# Módulo 7 · Clase 5 — Árboles de Decisión y Random Forest

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Árbol de Decisión: cómo funciona |
| 2 | Hiperparámetros clave: profundidad, min_samples |
| 3 | Importancia de features |
| 4 | Random Forest: el poder del ensemble |
| 5 | Comparativa de modelos |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, r2_score, classification_report

np.random.seed(42)
print("Librerías listas")

---
## 1. Árbol de Decisión: cómo funciona

```
Un árbol divide el espacio de features con preguntas binarias sucesivas:

  ¿edad < 30?
  ├── Sí → ¿ingresos < 20000?
  │         ├── Sí → NO APROBADO
  │         └── No → APROBADO
  └── No → ¿historial_limpio?
            ├── Sí → APROBADO
            └── No → NO APROBADO

¿Cómo elige qué pregunta hacer?
  Busca el corte que mejor separa las clases.
  Criterio: GINI o Entropía (clasificación), MSE (regresión)

GINI = 1 - Σpᵢ² (mide la impureza del nodo; 0 = puro, 0.5 = máximo mezcla)

Ventajas:
  ✓ Interpretable: se puede visualizar
  ✓ No necesita escalado de features
  ✓ Maneja features numéricas y categóricas
  ✓ Muy rápido

Desventajas:
  ✗ Overfitting si el árbol crece demasiado
  ✗ Inestable: pequeños cambios en datos → árbol muy diferente
```

In [ ]:
# Dataset Titanic simplificado
import seaborn as sns
df = sns.load_dataset('titanic')[['survived','pclass','sex','age','fare']].dropna()
df['sex_num'] = (df['sex'] == 'female').astype(int)
X = df[['pclass','sex_num','age','fare']]
y = df['survived']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Árbol con profundidad limitada (interpretable)
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_tr, y_tr)

print("Árbol de decisión (profundidad=3):")
print(export_text(arbol, feature_names=list(X.columns)))

print(f"Accuracy en test: {accuracy_score(y_te, arbol.predict(X_te)):.4f}")

# Visualizar el árbol
plt.figure(figsize=(15, 6))
plot_tree(arbol, feature_names=X.columns, class_names=['No sobrevivió','Sobrevivió'],
          filled=True, rounded=True, fontsize=9)
plt.title('Árbol de Decisión — Supervivencia Titanic (profundidad=3)')
plt.tight_layout()
plt.show()
print("\nLectura del árbol:")
print("  Nodo raíz: el corte más informativo es sex_num (sexo)")
print("  Nodos coloreados: azul = No sobrevivió, naranja = Sobrevivió")
print("  samples: nº de muestras en el nodo")
print("  value: distribución de clases [No, Sí]")

---
## 2. Hiperparámetros clave

In [ ]:
print("=== Efecto de max_depth ===")
print(f"{'max_depth':>12} {'R²_train':>10} {'R²_test':>10} {'Nodos':>8}")

for depth in [1, 2, 3, 5, 10, None]:
    arbol = DecisionTreeClassifier(max_depth=depth, random_state=42)
    arbol.fit(X_tr, y_tr)
    acc_tr = accuracy_score(y_tr, arbol.predict(X_tr))
    acc_te = accuracy_score(y_te, arbol.predict(X_te))
    n_nodos = arbol.tree_.node_count
    depth_str = str(depth) if depth else 'None (sin límite)'
    print(f"{depth_str:>12} {acc_tr:>10.4f} {acc_te:>10.4f} {n_nodos:>8}")

print("\nOtros hiperparámetros importantes:")
print("  min_samples_split: nº mínimo de muestras para hacer un corte (default=2)")
print("  min_samples_leaf: nº mínimo de muestras en un nodo hoja (default=1)")
print("  max_features: nº de features a considerar en cada corte")
print("  criterion: 'gini' (default) o 'entropy'")

# Visualización del efecto de la profundidad
np.random.seed(42)
X2 = np.random.randn(200, 2)
y2 = ((X2[:,0]**2 + X2[:,1]**2) > 1.2).astype(int)

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.3, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
x1_r = np.linspace(-3, 3, 100)
x2_r = np.linspace(-3, 3, 100)
xx, yy = np.meshgrid(x1_r, x2_r)
grid = np.c_[xx.ravel(), yy.ravel()]

for ax, depth, titulo in zip(axes, [1, 2, 5, None], ['depth=1', 'depth=2', 'depth=5', 'depth=None']):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X2_tr, y2_tr)
    Z = clf.predict(grid).reshape(xx.shape)
    acc = accuracy_score(y2_te, clf.predict(X2_te))
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X2_te[:,0], X2_te[:,1], c=y2_te, cmap='bwr', s=20, edgecolors='k', lw=0.3)
    ax.set_title(f'{titulo}\nAcc test={acc:.3f}')

plt.suptitle('Frontera de decisión según profundidad del árbol', fontsize=11)
plt.tight_layout(); plt.show()

---
## 3. Importancia de features

In [ ]:
arbol_full = DecisionTreeClassifier(max_depth=5, random_state=42)
arbol_full.fit(X_tr, y_tr)

importancias = pd.Series(arbol_full.feature_importances_, index=X.columns)
importancias = importancias.sort_values(ascending=True)

print("Importancia de features (árbol profundidad=5):")
for feat, imp in importancias.sort_values(ascending=False).items():
    bar = '█' * int(imp * 50)
    print(f"  {feat:15}: {imp:.4f}  {bar}")

importancias.plot(kind='barh', color='steelblue')
plt.xlabel('Importancia (reducción media de impureza)')
plt.title('Feature Importance — Titanic')
plt.tight_layout(); plt.show()

print("\nInterpretación:")
print("  Importancia = suma de la reducción de impureza (Gini) que produce la feature.")
print("  sex_num es la feature más importante → coincide con el análisis del EDA.")

---
## 4. Random Forest

```
PROBLEMA del árbol individual: overfitting e inestabilidad.

SOLUCIÓN — Random Forest (Ensemble de árboles):

  1. BAGGING: entrenar N árboles, cada uno con una muestra aleatoria
     con reemplazo de los datos de entrenamiento.

  2. RANDOM SUBSPACE: en cada nodo, solo se consideran
     sqrt(p) features (clasificación) o p/3 (regresión) aleatorias.
     → Los árboles son DISTINTOS entre sí → menos correlación → mejor ensemble.

  3. VOTING: para clasificar, se cuentan los votos de todos los árboles.
             para regresión, se hace la media.

Resultado: mucho más robusto al overfitting que un árbol individual.

Hiperparámetros clave:
  n_estimators: nº de árboles (más es mejor hasta cierto punto, luego plateau)
  max_depth:    profundidad de cada árbol (None por defecto)
  max_features: 'sqrt' (clasificación), 1.0 (regresión) por defecto
  min_samples_leaf: mínimo muestras por hoja
```

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

print("=== Comparativa: Árbol vs Random Forest ===")
print(f"{'Modelo':30} {'Acc_train':>10} {'Acc_test':>10} {'F1_test':>10}")

for nombre, m in [
    ('Árbol depth=3',       DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('Árbol depth=10',      DecisionTreeClassifier(max_depth=10, random_state=42)),
    ('Árbol sin límite',    DecisionTreeClassifier(random_state=42)),
    ('RandomForest 100',    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('RandomForest 500',    RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ('LogisticRegression',  LogisticRegression(max_iter=1000, random_state=42)),
]:
    m.fit(X_tr, y_tr)
    acc_tr = accuracy_score(y_tr, m.predict(X_tr))
    acc_te = accuracy_score(y_te, m.predict(X_te))
    f1_te  = f1_score(y_te, m.predict(X_te))
    print(f"{nombre:30} {acc_tr:>10.4f} {acc_te:>10.4f} {f1_te:>10.4f}")

# Efecto del nº de árboles
n_trees_range = [1, 5, 10, 20, 50, 100, 200]
acc_tests = []
for n in n_trees_range:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf_n.fit(X_tr, y_tr)
    acc_tests.append(accuracy_score(y_te, rf_n.predict(X_te)))

plt.figure(figsize=(8, 4))
plt.semilogx(n_trees_range, acc_tests, 'b-o', lw=2)
plt.xlabel('Nº de árboles (escala log)'); plt.ylabel('Accuracy en test')
plt.title('Random Forest: accuracy vs nº de árboles')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("A partir de ~100 árboles el beneficio marginal es pequeño.")

# Feature importance con RF
imp_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
imp_rf.plot(kind='barh', color='coral')
plt.xlabel('Importancia'); plt.title('Feature Importance — Random Forest')
plt.tight_layout(); plt.show()

---
## Práctica

### Ejercicio 1 — Árbol de Decisión en Iris

**Enunciado:** Usando el dataset Iris:

1. Entrena un árbol de profundidad 2, 4 y sin límite
2. Para cada uno: accuracy en train y test, número de hojas, número de nodos
3. Visualiza el árbol de profundidad 3
4. ¿A qué profundidad empieza el overfitting?
5. ¿Qué feature elige el árbol en el nodo raíz? ¿Coincide con la correlación?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

iris = datasets.load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"{'Depth':>8} {'Acc_train':>10} {'Acc_test':>10} {'Nodos':>8} {'Hojas':>8}")
for depth in [2, 3, 4, 6, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_tr, y_tr)
    print(f"{str(depth):>8} {accuracy_score(y_tr,t.predict(X_tr)):>10.4f} "
          f"{accuracy_score(y_te,t.predict(X_te)):>10.4f} "
          f"{t.tree_.node_count:>8} {t.get_n_leaves():>8}")

# Árbol profundidad 3
t3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
plt.figure(figsize=(14,6))
plot_tree(t3, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True, fontsize=9)
plt.title('Árbol de decisión Iris — profundidad 3'); plt.tight_layout(); plt.show()

feat_raiz = iris.feature_names[t3.tree_.feature[0]]
print(f"\nFeature en el nodo raíz: {feat_raiz}")
print("Esto coincide con la feature de mayor varianza entre clases.")
print("El overfitting empieza claramente a partir de depth=5-6.")
```
</details>

---
### Ejercicio 2 — Random Forest Regressor

**Enunciado:** Aplica Random Forest a un problema de regresión:

```python
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
```

1. Divide train/test (80/20)
2. Compara: LinearRegression, DecisionTreeRegressor (depth=5), RandomForestRegressor (100 árboles)
3. Métrica: RMSE y R² en test
4. Muestra las 5 features más importantes según Random Forest
5. ¿Por qué Random Forest supera a LinearRegression aquí?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

housing = fetch_california_housing()
X, y = housing.data, housing.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

modelos = {
    'LinearRegression':    LinearRegression(),
    'DecisionTree d=5':    DecisionTreeRegressor(max_depth=5, random_state=42),
    'RandomForest 100':    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}

print(f"{'Modelo':25} {'RMSE':>10} {'R²':>10}")
rf_modelo = None
for nombre, m in modelos.items():
    X_tr_fit = X_tr_s if 'Linear' in nombre else X_tr
    X_te_fit = X_te_s if 'Linear' in nombre else X_te
    m.fit(X_tr_fit, y_tr)
    y_pred = m.predict(X_te_fit)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)
    print(f"{nombre:25} {rmse:>10.4f} {r2:>10.4f}")
    if 'Random' in nombre: rf_modelo = m

# Feature importance
imp = sorted(zip(housing.feature_names, rf_modelo.feature_importances_),
             key=lambda x: x[1], reverse=True)
print("\nTop 5 features (Random Forest):")
for feat, imp_val in imp[:5]:
    print(f"  {feat:25}: {imp_val:.4f}")

print("\nRandom Forest supera a LinearRegression porque:")
print("  Las relaciones entre features y precio no son lineales.")
print("  Los árboles capturan interacciones y no-linealidades automáticamente.")
```
</details>

---
### Ejercicio 3 — Selección de features con importancia

**Enunciado:** Usa la importancia de features para seleccionar un subconjunto:

```python
np.random.seed(42)
n, p = 500, 20
X = np.random.randn(n, p)
# Solo 4 features importan
coef = np.zeros(p); coef[0]=2; coef[3]=-1.5; coef[7]=1; coef[12]=-0.8
y = (X @ coef + np.random.randn(n) > 0).astype(int)
```

1. Entrena RandomForestClassifier con todas las features
2. Ordena features por importancia y visualiza
3. Reentrena con solo las top 4 features
4. Compara accuracy con todas las features vs solo las importantes
5. ¿Recupera el RF las 4 features relevantes?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
n, p = 500, 20
X = np.random.randn(n, p)
coef = np.zeros(p); coef[0]=2; coef[3]=-1.5; coef[7]=1; coef[12]=-0.8
y = (X @ coef + np.random.randn(n) > 0).astype(int)

nombres = [f'feat_{i:02d}' for i in range(p)]
features_reales = ['feat_00', 'feat_03', 'feat_07', 'feat_12']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. RF completo
rf_full = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_full.fit(X_tr, y_tr)
acc_full = accuracy_score(y_te, rf_full.predict(X_te))

# 2. Ordenar por importancia
imp = list(zip(nombres, rf_full.feature_importances_))
imp.sort(key=lambda x: x[1], reverse=True)
print("Top 10 features:")
for feat, imp_val in imp[:10]:
    marca = '★' if feat in features_reales else '  '
    print(f"  {marca} {feat}: {imp_val:.4f}")

# Visualización
nombres_ord, imps_ord = zip(*imp)
colores = ['green' if f in features_reales else 'steelblue' for f in nombres_ord]
plt.figure(figsize=(12,4))
plt.bar(nombres_ord, imps_ord, color=colores)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title('Feature Importance (verde = relevante real)'); plt.tight_layout(); plt.show()

# 3. RF con top 4
top4_idx = [int(f.split('_')[1]) for f, _ in imp[:4]]
rf_top4 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_top4.fit(X_tr[:, top4_idx], y_tr)
acc_top4 = accuracy_score(y_te, rf_top4.predict(X_te[:, top4_idx]))

print(f"\nAccuracy todas las features: {acc_full:.4f}")
print(f"Accuracy top 4 features:     {acc_top4:.4f}")
print(f"\nFeatures top 4 detectadas: {[nombres_ord[i] for i in range(4)]}")
print(f"Features relevantes reales: {features_reales}")
```
</details>

---
### Ejercicio 4 — Comparativa completa de modelos

**Enunciado:** Sobre el dataset de detección de fraude:

```python
np.random.seed(42)
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=2000, n_features=15, n_informative=8,
                           weights=[0.9, 0.1], random_state=42)
```

1. Entrena: LogisticRegression, DecisionTree (depth=5), RandomForest (100), RandomForest (500)
2. Para cada modelo: Accuracy, F1, AUC-ROC en test
3. Visualiza las 4 curvas ROC en un mismo gráfico
4. ¿Qué modelo es mejor? ¿Por qué?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve

X, y = make_classification(n_samples=2000, n_features=15, n_informative=8,
                            weights=[0.9, 0.1], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler(); X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)

modelos = {
    'LogisticRegression': (LogisticRegression(class_weight='balanced', max_iter=1000), X_tr_s, X_te_s),
    'DecisionTree d=5':   (DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42), X_tr, X_te),
    'RandomForest 100':   (RandomForestClassifier(100, class_weight='balanced', random_state=42, n_jobs=-1), X_tr, X_te),
    'RandomForest 500':   (RandomForestClassifier(500, class_weight='balanced', random_state=42, n_jobs=-1), X_tr, X_te),
}

fig, ax = plt.subplots(figsize=(8, 6))
print(f"{'Modelo':22} {'Accuracy':>10} {'F1':>10} {'AUC':>10}")
for nombre, (m, Xtr, Xte) in modelos.items():
    m.fit(Xtr, y_tr)
    y_pred = m.predict(Xte)
    prob = m.predict_proba(Xte)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, prob)
    print(f"{nombre:22} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f}")
    fpr, tpr, _ = roc_curve(y_te, prob)
    ax.plot(fpr, tpr, lw=2, label=f'{nombre} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Curvas ROC comparadas'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("\nRandom Forest supera gracias a la combinación de árboles: reduce varianza sin aumentar sesgo.")
```
</details>

---
### Ejercicio 5 (reto) — Tuning manual de Random Forest

**Enunciado:** Implementa una búsqueda manual de hiperparámetros para Random Forest:

```python
param_grid = {
    'n_estimators':   [50, 100, 200],
    'max_depth':      [5, 10, None],
    'min_samples_leaf': [1, 5, 10],
}
```

1. Prueba todas las combinaciones (3×3×3 = 27)
2. Para cada combinación: F1 en test
3. Muestra las 5 mejores combinaciones
4. ¿Cuál es el mejor modelo?
5. ¿Por qué esta búsqueda manual tiene un problema? (pista: data leakage de hiperparámetros)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from itertools import product

X, y = make_classification(n_samples=2000, n_features=15, n_informative=8,
                            weights=[0.9, 0.1], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_grid = {
    'n_estimators':     [50, 100, 200],
    'max_depth':        [5, 10, None],
    'min_samples_leaf': [1, 5, 10],
}

resultados = []
combos = list(product(param_grid['n_estimators'],
                      param_grid['max_depth'],
                      param_grid['min_samples_leaf']))
print(f"Evaluando {len(combos)} combinaciones...")

for n, d, leaf in combos:
    rf = RandomForestClassifier(n_estimators=n, max_depth=d, min_samples_leaf=leaf,
                                class_weight='balanced', random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    f1 = f1_score(y_te, rf.predict(X_te))
    resultados.append({'n_estimators':n, 'max_depth':d, 'min_samples_leaf':leaf, 'f1_test':f1})

import pandas as pd
res_df = pd.DataFrame(resultados).sort_values('f1_test', ascending=False)
print("\nTop 5 combinaciones:")
print(res_df.head().to_string(index=False))

print("\n⚠️  PROBLEMA: evaluamos hiperparámetros en el mismo test set.")
print("   → Los hiperparámetros se 'sobreajustan' al test set.")
print("   → En producción, el rendimiento real puede ser peor.")
print("   SOLUCIÓN correcta: usar cross-validation (C7) para elegir hiperparámetros.")
print("   El test set final debe usarse UNA SOLA VEZ, al final.")
```
</details>

---
## Resumen

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Árbol de decisión
arbol = DecisionTreeClassifier(
    max_depth=5,           # limita el overfitting
    min_samples_leaf=5,    # mínimo muestras por hoja
    criterion='gini',
    random_state=42
)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=100,      # nº de árboles
    max_depth=None,        # árboles profundos + bagging = OK
    max_features='sqrt',   # features por nodo
    class_weight='balanced',  # para datos desbalanceados
    n_jobs=-1,             # usar todos los cores
    random_state=42
)

# Feature importance (igual en árbol y RF)
importancias = rf.feature_importances_
```

## ¿Qué viene en la próxima clase?
En la **Clase 6** entendemos el **overfitting** en profundidad: bias-variance tradeoff, curvas de aprendizaje y cómo diagnosticar si tu modelo tiene un problema de underfitting o overfitting.